In [1]:
"""
diag_replay.py -- localise a golden-gate FAIL. Compares the batch feature path
against the worker Engine, channel by channel, for one session.

Order of blame:
  1. leg_dir / leg_start_jma      -- state parity (Stage-0 rules)
  2. z-channels                   -- numerical (_expanding_z vs Welford)
  3. events                       -- grammar parity (which class fired when)

Run diagnose_channels() first. If every channel matches to ~1e-12 the cause is
grammar, and diagnose_events() names it.
"""

import numpy as np
import pandas as pd

from worker_diag import (Engine, Welford, load_contract, MODEL_PATH, SC_TAG,
                    load_research_frame, ZWARM, TAUS)
from common import _expanding_z

# ---------------------------------------------------------------- CONFIG
BARS_PATH = f"stage-0/bars_{SC_TAG}.pqt"        # Stage-0 output for this tag
EVENTS_PATH = f"stage-0/events_{SC_TAG}.pqt"
# ----------------------------------------------------------------


def _session(src, bars, day):
    g = (src[src["timestamp"].dt.date == day]
         .sort_values("timestamp").reset_index(drop=True))
    b = (bars[bars["date"].astype(str) == str(day)]
         .sort_values("bar_index").reset_index(drop=True))
    assert len(g) == len(b), f"row mismatch src={len(g)} bars={len(b)}"
    assert np.array_equal(g["JMA"].to_numpy(), b["jma"].to_numpy()), \
        "JMA differs between source file and stage-0 bars"
    return g, b


def _batch_state(b):
    """Exactly augment_featurizer: leg_dir + leg_start_jma from stage-0 bars."""
    jma = b["jma"].to_numpy(np.float64)
    leg_dir = b["jma_leg_dir"].to_numpy(np.float64)
    tgt = b["is_target"].to_numpy()
    starts = np.where(tgt)[0]
    leg_id = np.cumsum(tgt)
    start_idx = np.concatenate(([0], starts))[leg_id]
    return leg_dir, jma, jma[start_idx]


def _worker_state(c, g):
    """Drive the Engine over the session, capturing its internal state per bar."""
    cols = ["rawOpen", "rawLast", "JMA"] + c["stream_cols"]
    V = g[cols].to_numpy(np.float64)
    eng = Engine(c)
    n = len(g)
    leg_dir = np.zeros(n)
    leg_start = np.zeros(n)
    fired = []                                   # (local_bar, class) worker recorded
    for i in range(n):
        before = {cl: eng.last_event[cl] for cl in c["classes"]}
        eng.ingest(i, *V[i])
        leg_dir[i] = eng.jma_track.sign
        leg_start[i] = eng.leg_start_jma
        for cl in c["classes"]:
            if eng.last_event[cl] != before[cl]:
                fired.append((i, cl))
    return leg_dir, leg_start, fired


def diagnose_channels(day, src=None, bars=None):
    c = load_contract(MODEL_PATH)
    src = load_research_frame(c) if src is None else src
    bars = pd.read_parquet(BARS_PATH) if bars is None else bars
    g, b = _session(src, bars, day)

    bd, jma, bstart = _batch_state(b)
    wd, wstart, _ = _worker_state(c, g)

    print(f"==== {day}  ({len(g)} bars) ====")
    nd = int((bd != wd).sum())
    ns = int((np.abs(bstart - wstart) > 0).sum())
    print(f"leg_dir       mismatch bars: {nd}"
          + (f"   first at {int(np.argmax(bd != wd))}" if nd else ""))
    print(f"leg_start_jma mismatch bars: {ns}"
          + (f"   first at {int(np.argmax(np.abs(bstart - wstart) > 0))}" if ns else ""))
    if nd or ns:
        print("STATE PARITY BROKEN -- fix before reading channels below")

    osc = [g[col].to_numpy(np.float64) for col in c["stream_cols"]]
    body = g["rawLast"].to_numpy(np.float64) - g["rawOpen"].to_numpy(np.float64)

    chans = {}
    for i, col in enumerate(c["stream_cols"]):
        chans[f"z_{col}_signed"] = osc[i] * bd
    for i, col in enumerate(c["stream_cols"]):
        chans[f"z_{col}_mag"] = np.abs(osc[i])
    chans["z_leg_amp"] = np.abs(jma - bstart)
    chans["z_body_raw_signed"] = body * bd
    chans["z_body_raw_mag"] = np.abs(body)

    print(f"{'channel':24s} {'max|d|':>10s} {'n>1e-9':>8s} {'n>1e-6':>8s} {'worst':>7s}")
    rows = []
    for name, x in chans.items():
        zb = _expanding_z(x, ZWARM)
        w = Welford()
        zw = np.array([w.z_then_update(v) for v in x])
        D = np.abs(zb - zw)
        rows.append((name, float(D.max()), int((D > 1e-9).sum()),
                     int((D > 1e-6).sum()), int(np.argmax(D))))
        print(f"{name:24s} {D.max():10.3e} {int((D > 1e-9).sum()):8d} "
              f"{int((D > 1e-6).sum()):8d} {int(np.argmax(D)):7d}")
    return pd.DataFrame(rows, columns=["channel", "max_abs_d", "n_gt_1e9",
                                       "n_gt_1e6", "worst_bar"])


def diagnose_events(day, src=None, bars=None, events=None):
    """Compare which (class, local_bar) the worker records vs stage-0 events."""
    c = load_contract(MODEL_PATH)
    src = load_research_frame(c) if src is None else src
    bars = pd.read_parquet(BARS_PATH) if bars is None else bars
    events = pd.read_parquet(EVENTS_PATH) if events is None else events
    g, b = _session(src, bars, day)
    _, _, fired = _worker_state(c, g)

    first = int(b["bar_index"].iloc[0])
    e = events[events["date"].astype(str) == str(day)].copy()
    e["local"] = e["event_bar"].to_numpy() - first

    batch = set()
    for _, r in e.iterrows():
        if r["stream"] == c["self"]:
            batch.add((int(r["local"]), (c["self"], "all")))
        elif r["opposing"] != 0:
            batch.add((int(r["local"]),
                       (r["stream"], "opp" if r["opposing"] == 1 else "conf")))
    work = set(fired)

    only_w = sorted(work - batch)
    only_b = sorted(batch - work)
    print(f"==== {day} events ====")
    print(f"batch {len(batch)}   worker {len(work)}   "
          f"worker-only {len(only_w)}   batch-only {len(only_b)}")
    for tag, lst in (("worker-only", only_w), ("batch-only", only_b)):
        for bar, cl in lst[:15]:
            print(f"  {tag:12s} local_bar={bar:5d}  {cl}")
    return only_w, only_b

In [2]:
# ---------------------------------------------------------------- usage

DAY = pd.Timestamp("2026-03-05").date()
ch = diagnose_channels(DAY)
ow, ob = diagnose_events(DAY)

==== 2026-03-05  (3600 bars) ====
leg_dir       mismatch bars: 0
leg_start_jma mismatch bars: 0
channel                      max|d|   n>1e-9   n>1e-6   worst
z_jmaD1_signed            6.750e-14        0        0    3082
z_jmaD2_signed            7.105e-15        0        0     602
z_jmaD1_mag               7.283e-14        0        0    3082
z_jmaD2_mag               8.260e-14        0        0    3082
z_leg_amp                 5.329e-15        0        0     963
z_body_raw_signed         4.441e-15        0        0    2720
z_body_raw_mag            7.994e-15        0        0    2775
==== 2026-03-05 events ====
batch 2120   worker 2120   worker-only 0   batch-only 0


In [3]:
print(bad.date.unique()[:10])
DAY = bad.date.iloc[0]
ch = diagnose_channels(DAY)
ow, ob = diagnose_events(DAY)

NameError: name 'bad' is not defined